# 02 — Recherche du meilleur clustering


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd().resolve()
while not (ROOT / "datasets").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "datasets" / "ml-research"
df = pd.read_csv(DATA / "plants_SYNTHETIC_DEMO.csv")
print("Synthetic demo — NOT horticultural evidence:", df.shape)
df.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
NUM = ["height_cm", "spread_cm", "hardiness_min_c"]
CAT = ["plant_type", "evergreen", "sun_exposure", "soil_type"]
pre = ColumnTransformer([
    ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("category", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore", sparse_output=False)), CAT)
])
X = pre.fit_transform(df[NUM + CAT])
print(X.shape)

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances
results = []
for k in range(2, 9):
    for name, model in [("kmeans", KMeans(n_clusters=k, n_init=10, random_state=42)),
                        ("agglomerative", AgglomerativeClustering(n_clusters=k))]:
        labels = model.fit_predict(X)
        score = silhouette_score(X, labels)
        results.append({"algorithm":name,"parameter":k,"silhouette":score,"clusters":len(set(labels))})
for eps in [1.0, 2.0, 3.0, 4.0]:
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X)
    mask = labels != -1
    if len(set(labels[mask])) >= 2 and mask.sum() > len(set(labels[mask])):
        results.append({"algorithm":"dbscan","parameter":eps,
                        "silhouette":silhouette_score(X[mask], labels[mask]),
                        "clusters":len(set(labels[mask]))})
scores = pd.DataFrame(results).sort_values("silhouette",ascending=False)
display(scores.head(12))
scores.to_csv(DATA / "clustering_scores.csv",index=False)

In [ ]:
# The silhouette score alone does not establish horticultural usefulness.
# Inspect cluster profiles and stability before choosing an algorithm.